# Difference-in-Differences (DiD) Econometric Analysis
**Project:** Salon Experiment & Causal Analysis – Measuring the True Impact of Business Changes  
**Policy Intervention:** Centralized Regional Quality & Automated Service Guarantee Policy (Rolled out 2023-01)  
**Treated Units:** Central & North Coimbatore Regions  
**Control Units:** East, South, & West Coimbatore Regions  
**Data Scope:** 48 Monthly Regional Panel Observations (2021-01 to 2024-12, N = 240)

---

### Ethical Simulation Disclosure
> **NOTICE:** All data and regional panel observations in this notebook are synthetically generated for educational and causal evaluation. They do not represent proprietary company data.


## 1. Econometric Framework & Policy Background
In January 2023 (Month 25), Aura Luxe Salon implemented a regional management overhaul and service guarantee in its **Central** and **North** Coimbatore regions. To measure its causal effect without randomized rollout, we formulate a two-way Difference-in-Differences (DiD) model:

$$Y_{rt} = \beta_0 + \beta_1(\text{Treatment}_r) + \beta_2(\text{Post}_t) + \beta_3(\text{Treatment}_r \times \text{Post}_t) + \epsilon_{rt}$$

Where $\beta_3$ represents the average treatment effect on the treated (ATT).


In [1]:
import os
import sys
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath("../src"))
from causal_inference import fit_did_model, fit_event_study_model

DATA_DIR = os.path.abspath("../data/processed")
df_hist = pd.read_csv(os.path.join(DATA_DIR, "historical_regional_data_clean.csv"))

print("Historical regional panel records:", len(df_hist))
display(df_hist.head())


Historical regional panel records: 240
     month   region  treatment_region_indicator  post_intervention_indicator  revenue_per_visit  customer_retention  no_show_rate  number_of_appointments
0  2021-01  Central                           1                            0            1620.54              0.6765        0.0774                    8683
1  2021-01    North                           1                            0            1475.45              0.5982        0.0995                    6786
2  2021-01     East                           0                            0            1347.46              0.6044        0.1041                    7187
3  2021-01    South                           0                            0            1235.56              0.6003        0.1116                    5736
4  2021-01     West                           0                            0            1308.27              0.6311        0.1064                    4887


## 2. DiD Regression Estimation (Cluster-Robust Standard Errors)
We estimate the treatment effect across three key regional outcomes: Revenue per visit, Customer retention, and No-show rate.


In [2]:
did_rev = fit_did_model(df_hist, "revenue_per_visit")
did_ret = fit_did_model(df_hist, "customer_retention")
did_ns = fit_did_model(df_hist, "no_show_rate")

did_table = pd.DataFrame([
    {
        "Outcome": "Revenue per Visit (INR)",
        "Counterfactual Control": f"Rs. {did_rev['post_treatment_control_mean']:.2f}",
        "DiD Coef (beta3)": f"Rs. {did_rev['did_coefficient_beta3']:+.2f}",
        "Cluster SE": f"Rs. {did_rev['standard_error']:.2f}",
        "95% CI": f"[Rs. {did_rev['ci_lower_95']:+.2f}, Rs. {did_rev['ci_upper_95']:+.2f}]",
        "p-value": f"{did_rev['p_value']:.4e}",
        "Significant": did_rev["statistically_significant"]
    },
    {
        "Outcome": "Customer Retention Rate",
        "Counterfactual Control": f"{did_ret['post_treatment_control_mean']*100:.2f}%",
        "DiD Coef (beta3)": f"{did_ret['did_coefficient_beta3']*100:+.2f} pp",
        "Cluster SE": f"{did_ret['standard_error']*100:.2f} pp",
        "95% CI": f"[{did_ret['ci_lower_95']*100:+.2f} pp, {did_ret['ci_upper_95']*100:+.2f} pp]",
        "p-value": f"{did_ret['p_value']:.4e}",
        "Significant": did_ret["statistically_significant"]
    },
    {
        "Outcome": "No-Show Rate",
        "Counterfactual Control": f"{did_ns['post_treatment_control_mean']*100:.2f}%",
        "DiD Coef (beta3)": f"{did_ns['did_coefficient_beta3']*100:+.2f} pp",
        "Cluster SE": f"{did_ns['standard_error']*100:.2f} pp",
        "95% CI": f"[{did_ns['ci_lower_95']*100:+.2f} pp, {did_ns['ci_upper_95']*100:+.2f} pp]",
        "p-value": f"{did_ns['p_value']:.4e}",
        "Significant": did_ns["statistically_significant"]
    }
])
display(did_table)


                   Outcome Counterfactual Control DiD Coef (beta3) Cluster SE                      95% CI      p-value  Significant
0  Revenue per Visit (INR)            Rs. 1409.87      Rs. +175.16   Rs. 6.81  [Rs. +161.80, Rs. +188.52]  1.0810e-145         True
1  Customer Retention Rate                 56.19%         +4.61 pp    0.21 pp        [+4.21 pp, +5.02 pp]  4.4189e-111         True
2             No-Show Rate                 11.85%         -3.36 pp    0.08 pp        [-3.52 pp, -3.19 pp]   0.0000e+00         True


## 3. Parallel Trends Assumption & Event-Study Dynamics
The core identifying assumption of DiD is that treatment and control regions would have followed parallel trends in the absence of the policy. We evaluate pre-intervention trends and fit an Event-Study regression with reference month $t = -1$ (2022-12).


In [3]:
df_es = fit_event_study_model(df_hist, outcome_col="revenue_per_visit")
print("--- Event-Study Dynamic Coefficients (Selection of Leads & Lags) ---")
display(df_es.iloc[20:30])

# Pre-trend hypothesis test
pre_leads = df_es[df_es["period_type"] == "Pre-Intervention Lead"]
pre_pvals = pre_leads["p_value"]
print(f"Pre-intervention lead coefficients count: {len(pre_leads)}")
print(f"Proportion of pre-intervention coefficients indistinguishable from zero (p >= 0.05): {(pre_pvals >= 0.05).mean()*100:.1f}%")


--- Event-Study Dynamic Coefficients (Selection of Leads & Lags) ---
    rel_time calendar_month  coefficient  standard_error  ci_lower_95  ci_upper_95  p_value            period_type
20        -4        2022-09     -25.3150         46.3486    -116.1565      65.5265   0.5849  Pre-Intervention Lead
21        -3        2022-10      -1.2850         20.5355     -41.5338      38.9638   0.9501  Pre-Intervention Lead
22        -2        2022-11       2.8567         34.5602     -64.8800      70.5934   0.9341  Pre-Intervention Lead
23        -1        2022-12       0.0000          0.0000       0.0000       0.0000   1.0000       Reference (Base)
24         0        2023-01     159.6183         35.8332      89.3866     229.8500   0.0000  Post-Intervention Lag
25         1        2023-02     175.7317         63.7714      50.7421     300.7213   0.0059  Post-Intervention Lag
26         2        2023-03     152.6917         39.0404      76.1739     229.2094   0.0001  Post-Intervention Lag
27         

## 4. Econometric Limitations & Clustered Standard Errors Discussion
### Critical Econometric Considerations:
1. **Small Cluster Problem ($G = 5$ Regions):** As emphasized by Cameron & Miller (2015), cluster-robust variance estimators can exhibit downward bias when the number of clusters is small ($G < 30$). While clustering at the region level accounts for serial correlation within geographic clusters, asymptotic assumptions may be fragile.
2. **Parallel Trends Non-Rejection Caveat:** A non-significant test on pre-intervention leads does not mathematically *prove* parallel trends; it merely indicates absence of detectable pre-divergence under the observed sample power.
3. **Macroeconomic Confounders:** Seasonal festivals (Diwali in Oct/Nov, Pongal in Jan) produce large seasonal shifts. The two-way fixed effects structure explicitly controls for common time shocks, preserving causal interpretability.
